# Pipelines in Machine Learning

### Importing Library

In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeClassifier

### Loading dataset

In [9]:
df = pd.read_csv('E:/Dataset/titanic-Dataset.csv')
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
26,27,0,3,"Emir, Mr. Farred Chehab",male,NaN,0,0,2631,7.2250,NaN,C
404,405,0,3,"Oreskovic, Miss. Marija",female,20.0,0,0,315096,8.6625,NaN,S
528,529,0,3,"Salonen, Mr. Johan Werner",male,39.0,0,0,3101296,7.9250,NaN,S
323,324,1,2,"Caldwell, Mrs. Albert Francis (Sylvia Mae Harb...",female,22.0,1,1,248738,29.0000,NaN,S
463,464,0,2,"Milling, Mr. Jacob Christian",male,48.0,0,0,234360,13.0000,NaN,S


### Droping unused columns

In [11]:
df.drop(columns=['PassengerId','Name','Ticket','Cabin'], inplace=True)

In [12]:
df.sample(4)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
414,1,3,male,44.0,0,0,7.9250,S
883,0,2,male,28.0,0,0,10.5000,S
103,0,3,male,33.0,0,0,8.6542,S
770,0,3,male,24.0,0,0,9.5000,S


### Splitting the dataset into train, and test

In [13]:
x_train,x_test,y_train,y_test = train_test_split(df.drop(columns=['Survived']), df['Survived'], test_size=0.2, random_state=12)

In [15]:
x_train.head(2)

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
8,3,female,27.0,0,2,11.1333,S
150,2,male,51.0,0,0,12.5250,S


### Checking for null

In [16]:
df.isnull().sum()

Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

### Since nullity is here, so using imputation

In [20]:
si_age = SimpleImputer()
si_embarked = SimpleImputer(strategy='most_frequent')

x_train_age = si_age.fit_transform(x_train[['Age']])
x_train_embarked = si_embarked.fit_transform(x_train[['Embarked']])

x_test_age = si_age.transform(x_test[['Age']])
x_test_embarked = si_embarked.transform(x_test[['Embarked']])

In [23]:
x_train_age.shape

(712, 1)

### Using OneHotEncoding on sex, and embarked column

In [24]:
ohe_sex = OneHotEncoder(sparse_output=False, handle_unknown='ignore')  # If another value came in future, then ignore it
ohe_embarked = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

x_train_sex = ohe_sex.fit_transform(x_train[['Sex']])
x_train_embarked = ohe_embarked.fit_transform(x_train_embarked)

x_test_sex = ohe_sex.transform(x_test[['Sex']])
x_test_embarked = ohe_embarked.transform(x_test_embarked)

In [28]:
x_train_embarked

array([[0., 0., 1.],
       [0., 0., 1.],
       [0., 0., 1.],
       ...,
       [0., 0., 1.],
       [0., 0., 1.],
       [1., 0., 0.]], shape=(712, 3))

### Now Concatenate all the columns of data

In [30]:
x_train.sample()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
587,1,male,60.0,1,1,79.2,C


In [31]:
x_train_rem = x_train.drop(columns=['Sex','Age','Embarked'])
x_test_rem = x_test.drop(columns=['Sex','Age','Embarked'])

In [33]:
x_train_transformed = np.concatenate((x_train_rem, x_train_sex, x_train_age, x_train_embarked), axis=1)
x_test_transformed = np.concatenate((x_test_rem, x_test_sex, x_test_age, x_test_embarked), axis=1)

In [34]:
x_train_transformed

array([[3., 0., 2., ..., 0., 0., 1.],
       [2., 0., 0., ..., 0., 0., 1.],
       [2., 0., 0., ..., 0., 0., 1.],
       ...,
       [1., 1., 2., ..., 0., 0., 1.],
       [3., 0., 0., ..., 0., 0., 1.],
       [3., 0., 0., ..., 1., 0., 0.]], shape=(712, 10))

### Visualize decision tree

In [36]:
clf = DecisionTreeClassifier()
clf.fit(x_train_transformed, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [37]:
y_pred = clf.predict(x_test_transformed)

In [39]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test, y_pred)

0.770949720670391

### Export your model

In [40]:
import pickle

In [42]:
pickle.dump(ohe_sex, open('models/ohe_sex.pkl', 'wb'))
pickle.dump(ohe_embarked, open('models/ohe_embarked.pkl', 'wb'))
pickle.dump(clf, open('models/clf.pkl', 'wb'))